### 🛠️ Phase 1: Delta Encoding & Decoding

Delta encoding is a lossy-to-lossless preprocessing technique widely used in data compression, especially effective for continuous data streams or gradient images.

#### 📌 Core Principle
Instead of storing absolute values, the encoder records the **difference (delta) between the current byte and the previous byte**.
* **Encoding Formula**: $D_i = (X_i - X_{i-1}) \pmod{256}$
* **Decoding Formula**: $X_i = (D_i + X_{i-1}) \pmod{256}$

#### 🌟 Design Highlight: Modular Arithmetic Defense
* **Overflow/Underflow Prevention**: By applying the `% 256` modulo operation, the data is **guaranteed to stay perfectly within the unsigned 8-bit byte range (`0 ~ 255`)** even when subtractions yield negative numbers or additions overflow. This completely eliminates hardware-level overflow bugs.

#### 🎯 Primary Objective
This preprocessing transforms varying data streams into **large clusters of repeating `0`s or uniform values**, drastically reducing the information entropy and smoothing the path for the subsequent LZ77 algorithm.

In [1]:
def delta_encode(data: bytes) -> bytearray:
    """Transforms continuous data into incremental byte-level differences (Deltas).
    
    This preprocessing layout reduces entropy by generating clusters of zeros,
    which optimizes the data stream for subsequent LZ77 window matching.
    """
    if not data:
        return bytearray()
    
    output = bytearray(len(data))
    output[0] = data[0] 
    
    for i in range(1, len(data)):
        # 🌟 WHY: Using modular arithmetic (% 256) ensures the delta wraps around seamlessly.
        # This keeps the result strictly within the unsigned 8-bit range (0-255) and prevents integer underflow.
        output[i] = (data[i] - data[i-1]) % 256
    return output

def delta_decode(data: bytearray) -> bytes:
    """Reconstructs the original byte stream from modulo-encoded delta differences."""
    if not data:
        return b""
    
    output = bytearray(len(data))
    output[0] = data[0]
    
    for i in range(1, len(data)):
        # 🌟 WHY: Reverses the encoding phase. Modulo 256 automatically resolves any previous underflow wraps.
        output[i] = (data[i] + output[i-1]) % 256
    return bytes(output)

### 📦 Phase 2: LZ77 Sliding Window Compression

This phase implements a Sliding Window dictionary-based compression algorithm, utilizing a Hash Map to accelerate repeated string matching.

#### ⚙️ Strict Boundary & Parameter Constraints
To ensure a seamless handoff to the downstream binary bit-packing stage, parameters are strictly capped to prevent bit-width overflow:
* **Search Window Size**: Rigidly locked at `3000` bytes. This ensures the offset value fits comfortably under the 12-bit maximum threshold of 4095, acting as a boundary defense.
* **Lookahead Buffer**: The maximum match length is strictly capped at `255` bytes, aligning perfectly with the 8-bit upper limit.

#### 🚀 Performance Optimization
* **Triple-Hash Acceleration**: A dictionary (`pos_hash`) tracks historical byte positions using a 3-byte tuple (`triple`) as the key. This drastically reduces the search complexity from $O(N^2)$ window scanning to near-constant lookups.

#### 📄 Output Token Format
The data stream is converted into a list of structurally unified tokens:
1. **Literal Token (No Match)**: `(False, literal_value, 0)`
2. **Reference Token (Match)**: `(True, distance, length)`

In [2]:
def lz77_compress(data: bytes, window_size: int = 3000) -> list:
    """Compresses data using a sliding window dictionary approach.
    
    Returns a list of structured tokens:
    - Literal: (False, byte_value, 0)
    - Reference Match: (True, distance, length)
    """
    tokens = []
    cursor = 0
    data_len = len(data)
    max_match_len = 255  # 🌟 WHY: Capped at 255 to perfectly fit into a single downstream 8-bit stream segment.
    
    pos_hash = {}
    
    while cursor < data_len:
        match_dist = 0
        match_len = 0
        
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            p = pos_hash.get(triple, -1)
            
            # 🌟 WHY: Enforces boundary controls. The historical position must reside inside the active 3000-byte window.
            if p != -1 and (cursor - p <= window_size) and (p < cursor):
                curr_match_len = 0
                while (cursor + curr_match_len < data_len and \
                       data[p + curr_match_len] == data[cursor + curr_match_len] and \
                       curr_match_len < max_match_len):
                    curr_match_len += 1
                    
                if curr_match_len >= 3:
                    match_len = curr_match_len
                    match_dist = cursor - p

        if match_len >= 3:
            # 🌟 WHY: Defensive check to ensure values do not exceed the architectural limits of 12-bit/8-bit bitstreams.
            if 0 < match_dist <= 4095 and 3 <= match_len <= 255:
                tokens.append((True, match_dist, match_len))
                if cursor + 3 <= data_len:
                    triple = (data[cursor], data[cursor+1], data[cursor+2])
                    pos_hash[triple] = cursor
                cursor += match_len
                continue
                
        tokens.append((False, data[cursor], 0))
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            pos_hash[triple] = cursor
        cursor += 1
            
    return tokens

def lz77_decompress(tokens: list) -> bytes:
    """Restores the raw byte sequences from LZ77 literal and reference tokens."""
    output = bytearray()
    for is_match, val, length in tokens:
        if not is_match:
            output.append(val)
        else:
            distance = val
            start_pos = len(output) - distance
            for i in range(length):
                output.append(output[start_pos + i])
    return bytes(output)

### 🌳 Phase 3: Two-Tree Canonical Huffman Coding & Dynamic Header Box Layout

To maximize spatial compression velocity and performance, Version 3 eliminates the use of fixed bit-width slots for LZ77 reference tracking. Instead, the entropy pipeline splits the data stream into two completely independent **Canonical Huffman Trees**.

#### 📌 Dual-Tree Mapping Mathematics & Symbol Boundaries
Symbols generated from the upstream preprocessing and dictionary steps are tracked within two distinct probability distributions:
1. **Tree A (Literal / Length Tree — Capped at 512 Symbols)**:
   * $\text{Symbol } 0 \sim 255$: Represents raw preprocessed literal byte values.
   * $\text{Symbol } 256$: Acts as an explicit End-of-Block (EOB) / EOF control character to halt the reading loop.
   * $\text{Symbol } 257 \sim 511$: Represents matched LZ77 reference lengths mapped linearly via: $\text{Symbol} = \text{Length} - 3 + 257$.
2. **Tree B (Distance Tree — Capped at 4096 Symbols)**:
   * $\text{Symbol } 0 \sim 4095$: Dedicated entirely to statistical variable-length coding of LZ77 sliding window distances.

#### ⚙️ Defensive Architecture: Dynamic Tail Truncation
Canonical Huffman mechanics ensure that **as long as the sequence of bit-lengths for each symbol is shared, both sides can reconstruct identical binary decode trees symmetrically without transferring parent/child tree nodes**. To minimize header bloat caused by non-existent symbols, this design implements dynamic truncation of unreferenced trailing elements.

| Field Name | Byte Length | Architectural Defense & Structural Logic |
| :--- | :--- | :--- |
| **Magic Number** | 2 Bytes (`b"MY"`) | Validates custom compression framework layout configuration. |
| **Original Size** | 4 Bytes Unsigned Int | Source file size (Little-Endian) to verify symmetric buffer allocations. |
| **Max Lit-Len Used**| 2 Bytes Unsigned Int | Captures the maximum index boundary $N_A$ utilized by Tree A. |
| **Tree A Lengths** | $N_A + 1$ Bytes | Linear array tracking the exact bit-lengths for Tree A elements. |
| **Max Dist Used** | 2 Bytes Unsigned Int | Captures the maximum index boundary $N_B$ utilized by Tree B. |
| **Tree B Lengths** | $N_B + 1$ Bytes | Linear array tracking the exact bit-lengths for Tree B elements. |

The interleaved payload bitstream begins directly at the end of the header structures. Whenever a reference token is flagged by a Tree A length symbol ($\ge 257$), the corresponding Tree B distance bitstream sequence is packed sequentially next to it.

In [3]:
import struct
import heapq
from collections import Counter

class BitWriter:
    """Manages an un-aligned bitstream, packing arbitrary bit-widths into standard bytes."""
    def __init__(self):
        self.bytes_data = bytearray()
        self.buffer = 0
        self.bit_count = 0

    def write_bits(self, value: int, num_bits: int):
        # 🛡️ DEFENSE: High-order mask cutoff prevents out-of-bounds parameter data leakage.
        value = value & ((1 << num_bits) - 1)
        for i in range(num_bits - 1, -1, -1):
            bit = (value >> i) & 1
            self.buffer = (self.buffer << 1) | bit
            self.bit_count += 1
            if self.bit_count == 8:
                self.bytes_data.append(self.buffer)
                self.buffer = 0
                self.bit_count = 0

    def flush(self) -> bytearray:
        # 🌟 WHY: Pads remaining bits by shifting left to ensure correct bit weights at EOF.
        if self.bit_count > 0:
            self.buffer = self.buffer << (8 - self.bit_count)
            self.bytes_data.append(self.buffer)
            self.buffer = 0
            self.bit_count = 0
        return self.bytes_data

class BitReader:
    """Parses sequential bits from a compressed binary stream payload."""
    def __init__(self, data: bytes):
        self.data = data
        self.byte_idx = 0
        self.bit_idx = 7

    def read_bit(self) -> int:
        if self.byte_idx >= len(self.data):
            return 0
        bit = (self.data[self.byte_idx] >> self.bit_idx) & 1
        self.bit_idx -= 1
        if self.bit_idx < 0:
            self.bit_idx = 7
            self.byte_idx += 1
        return bit

def get_huffman_lengths(frequencies: dict, max_symbols: int) -> dict:
    """Generates standard Huffman code lengths using an accelerated min-heap tree."""
    if len(frequencies) == 0:
        return {i: 0 for i in range(max_symbols)}
    
    # 🌟 WHY: Forces a dummy companion symbol if there is only one distinct value in the stream,
    # ensuring the min-heap can successfully construct a valid tree layout.
    if len(frequencies) == 1:
        sym = list(frequencies.keys())[0]
        dummy = (sym + 1) % max_symbols
        frequencies[dummy] = 1

    # 🛡️ DEFENSE: Appending sym to the list prevents internal tie-breaking object comparison exceptions.
    heap = [[wt, sym, [sym]] for sym, wt in frequencies.items()]
    heapq.heapify(heap)
    
    lengths = {sym: 0 for sym in frequencies}
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for sym in lo[2]: lengths[sym] += 1
        for sym in hi[2]: lengths[sym] += 1
        heapq.heappush(heap, [lo[0] + hi[0], min(lo[1], hi[1]), lo[2] + hi[2]])
    
    full_lengths = {i: 0 for i in range(max_symbols)}
    for sym, l in lengths.items():
        full_lengths[sym] = l
    return full_lengths

def generate_canonical_codes(code_lengths: dict) -> dict:
    """Calculates alphanumeric Canonical Huffman codes deterministically based solely on target lengths."""
    if not code_lengths:
        return {}
    
    max_len = max(code_lengths.values())
    bl_count = Counter(code_lengths.values())
    
    next_code = {}
    code = 0
    bl_count[0] = 0
    for bits in range(1, max_len + 1):
        code = (code + bl_count[bits - 1]) << 1
        next_code[bits] = code
        
    canonical_codes = {}
    # 🌟 WHY: Sorting symbols ensures symmetrical code assignment across encoding and decoding steps.
    for sym in sorted(code_lengths.keys()):
        length = code_lengths[sym]
        if length > 0:
            code_val = next_code[length]
            canonical_codes[sym] = format(code_val, f'0{length}b')
            next_code[length] += 1
            
    return canonical_codes

def my_custom_compress(original_data: bytes) -> bytes:
    """Encodes raw inputs using a multi-layered pipeline: Delta -> LZ77 -> Two-Tree Canonical Huffman."""
    if not original_data:
        return b""
    
    # 1. Pipeline preprocessing stages
    delta_data = delta_encode(original_data)
    lz77_tokens = lz77_compress(delta_data)
    
    # 2. Separate streams and gather frequency metrics
    lit_len_symbols = []
    dist_symbols = []
    freq_lit_len = Counter()
    freq_dist = Counter()
    
    for is_match, val, length in lz77_tokens:
        if not is_match:
            lit_len_symbols.append(val)
            freq_lit_len[val] += 1
        else:
            len_sym = length - 3 + 257
            lit_len_symbols.append(len_sym)
            freq_lit_len[len_sym] += 1
            
            dist_symbols.append(val)
            freq_dist[val] += 1
            
    # Record structural block terminator
    lit_len_symbols.append(256)
    freq_lit_len[256] += 1
    
    # 3. Calculate code tree lengths
    raw_lengths_a = get_huffman_lengths(freq_lit_len, max_symbols=512)
    raw_lengths_b = get_huffman_lengths(freq_dist, max_symbols=4096)
    
    # 4. Truncate trailing unused symbols to maximize header spatial efficiency
    max_lit_used = 0
    for sym in range(512):
        if raw_lengths_a[sym] > 0:
            max_lit_used = sym
            
    max_dist_used = 0
    for sym in range(4096):
        if raw_lengths_b[sym] > 0:
            max_dist_used = sym
            
    # 5. Pack binary Header Box layout
    header_box = bytearray()
    header_box += struct.pack("<2sI", b"MY", len(original_data))
    header_box += struct.pack("<H", max_lit_used)
    for sym in range(max_lit_used + 1):
        header_box.append(raw_lengths_a[sym])
        
    header_box += struct.pack("<H", max_dist_used)
    for sym in range(max_dist_used + 1):
        header_box.append(raw_lengths_b[sym])
        
    # 6. Initialize tables and output compressed stream loops
    huff_table_a = generate_canonical_codes({k: v for k, v in raw_lengths_a.items() if v > 0})
    huff_table_b = generate_canonical_codes({k: v for k, v in raw_lengths_b.items() if v > 0})
    
    writer = BitWriter()
    dist_ptr = 0
    
    for sym in lit_len_symbols:
        bit_str_a = huff_table_a[sym]
        writer.write_bits(int(bit_str_a, 2), len(bit_str_a))
        
        # 🌟 WHY: A symbol above 256 indicates a length value match flag.
        # The corresponding distance bitstream sequence must follow immediately after it.
        if sym > 256:
            dist_val = dist_symbols[dist_ptr]
            bit_str_b = huff_table_b[dist_val]
            writer.write_bits(int(bit_str_b, 2), len(bit_str_b))
            dist_ptr += 1
            
    return bytes(header_box) + writer.flush()

def my_custom_decompress(compressed_bytes: bytes) -> bytes:
    """Decodes a two-tree optimized byte stream using symmetrical table reconstruction."""
    if not compressed_bytes:
        return b""
    
    # 1. Parse header box bounds
    magic, orig_size = struct.unpack("<2sI", compressed_bytes[0:6])
    if magic != b"MY":
        raise ValueError("Invalid custom compressed file layout configuration.")
        
    # 2. Extract Tree A length table limits
    ptr = 6
    max_lit_used = struct.unpack("<H", compressed_bytes[ptr:ptr+2])[0]
    ptr += 2
    
    lengths_a = {i: 0 for i in range(512)}
    for sym in range(max_lit_used + 1):
        lengths_a[sym] = compressed_bytes[ptr]
        ptr += 1
        
    # 3. Extract Tree B length table limits
    max_dist_used = struct.unpack("<H", compressed_bytes[ptr:ptr+2])[0]
    ptr += 2
    
    lengths_b = {i: 0 for i in range(4096)}
    for sym in range(max_dist_used + 1):
        lengths_b[sym] = compressed_bytes[ptr]
        ptr += 1
        
    # 4. Invert maps to generate code-to-symbol tables
    codes_a = generate_canonical_codes({k: v for k, v in lengths_a.items() if v > 0})
    codes_b = generate_canonical_codes({k: v for k, v in lengths_b.items() if v > 0})
    
    dec_table_a = {v: k for k, v in codes_a.items()}
    dec_table_b = {v: k for k, v in codes_b.items()}
    
    # 5. Extract bit payload values
    reader = BitReader(compressed_bytes[ptr:])
    lz77_tokens = []
    
    while True:
        current_bits = ""
        sym_a = None
        while True:
            current_bits += str(reader.read_bit())
            if current_bits in dec_table_a:
                sym_a = dec_table_a[current_bits]
                break
            # 🛡️ DEFENSE: Aborts if bitstream corruption creates codes longer than physically possible.
            if len(current_bits) > 32:
                break
                
        if sym_a == 256 or sym_a is None:
            break
            
        if sym_a < 256:
            lz77_tokens.append((False, sym_a, 0))
        else:
            length = sym_a - 257 + 3
            
            bit_str_b = ""
            distance = None
            while True:
                bit_str_b += str(reader.read_bit())
                if bit_str_b in dec_table_b:
                    distance = dec_table_b[bit_str_b]
                    break
                if len(bit_str_b) > 32:
                    break
            
            lz77_tokens.append((True, distance, length))
            
    # 6. Execute reverse-pipeline recovery steps
    lz77_decoded = lz77_decompress(lz77_tokens)
    original_restored = delta_decode(bytearray(lz77_decoded))
    
    # 🛡️ DEFENSE: Validates length alignment to catch memory execution faults.
    if len(original_restored) != orig_size:
        raise ValueError("🛡️ BOUNDARY FAULT: Output dimensions mismatch recorded configuration parameters.")
        
    return original_restored

In [4]:
import os
import time
import datetime
import numpy as np
import pandas as pd
from collections import Counter

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
# Change these variables every time you upgrade your algorithm
CURRENT_VERSION = "v3_two_tree"
VERSION_NOTES = "Version 3: Multi-stage pipeline upgraded to dual discrete Canonical Huffman Trees (Literal/Length and Distance) featuring tail-truncated headers."

# Benchmark settings
HISTORY_FILE = "benchmark_history.csv"
NUM_RUNS = 5  # Number of runs to average out timing noise
TARGET_FILES = [
    "test1.txt",
    "test2.txt",
    "test3.txt",
    "Lenna.bmp",
    "Cameraman.bmp"
]

def run_benchmark_with_history():
    """
    Runs the benchmark for the current version, persists the data into a CSV file,
    and displays both the current run and historical comparison.
    """
    current_results = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    print(f"🚀 Starting Benchmark for Version: {CURRENT_VERSION}")
    print(f"📝 Notes: {VERSION_NOTES}")
    print(f"🔄 Repeating each test {NUM_RUNS} times to eliminate timing noise...\n")
    
    for file_name in TARGET_FILES:
        if not os.path.exists(file_name):
            print(f"⚠️ Warning: '{file_name}' not found. Skipped.")
            continue
            
        # 1. Read original data
        with open(file_name, "rb") as f:
            original_data = f.read()
            
        orig_size = len(original_data)
        if orig_size == 0:
            continue
            
        # 2. Benchmark Compression (Run multiple times for averaging)
        comp_times = []
        compressed_data = b""
        for _ in range(NUM_RUNS):
            t0 = time.perf_counter()
            compressed_data = my_custom_compress(original_data)
            t1 = time.perf_counter()
            comp_times.append((t1 - t0) * 1000) # Convert to ms
            
        comp_time_ms = np.mean(comp_times)
        comp_size = len(compressed_data)
        
        # 3. Benchmark Decompression (Run multiple times for averaging)
        decomp_times = []
        decompressed_data = b""
        for _ in range(NUM_RUNS):
            t2 = time.perf_counter()
            decompressed_data = my_custom_decompress(compressed_data)
            t3 = time.perf_counter()
            decomp_times.append((t3 - t2) * 1000) # Convert to ms
            
        decomp_time_ms = np.mean(decomp_times)
        
        # 4. Calculate Objective Metrics
        comp_ratio = orig_size / comp_size if comp_size > 0 else 0
        space_saving = ((orig_size - comp_size) / orig_size) * 100
        
        # Throughput in MB/s = (Bytes / 1024 / 1024) / (ms / 1000)
        orig_size_mb = orig_size / (1024 * 1024)
        comp_throughput = orig_size_mb / (comp_time_ms / 1000) if comp_time_ms > 0 else 0
        decomp_throughput = orig_size_mb / (decomp_time_ms / 1000) if decomp_time_ms > 0 else 0
        
        is_valid = "PASS" if decompressed_data == original_data else "FAIL"
        
        # 5. Store current run data
        current_results.append({
            "timestamp": timestamp,
            "version": CURRENT_VERSION,
            "notes": VERSION_NOTES,
            "file_name": file_name,
            "orig_size_bytes": orig_size,
            "comp_size_bytes": comp_size,
            "comp_time_ms": round(comp_time_ms, 2),
            "decomp_time_ms": round(decomp_time_ms, 2),
            "comp_ratio": round(comp_ratio, 2),
            "space_saving_pct": round(space_saving, 2),
            "comp_throughput_mbs": round(comp_throughput, 2),
            "decomp_throughput_mbs": round(decomp_throughput, 2),
            "verification": is_valid
        })

    # Create DataFrame for the current run
    df_current = pd.DataFrame(current_results)
    
    # ==========================================
    # PERSISTENCE (SAVE TO CSV)
    # ==========================================
    if os.path.exists(HISTORY_FILE):
        df_history = pd.read_csv(HISTORY_FILE)
        # Prevent appending duplicate entries if the block is re-run with the same version on the exact same files
        # We drop existing logs for the same version and file to keep the history clean
        df_history = df_history[~((df_history["version"] == CURRENT_VERSION) & (df_history["file_name"].isin(df_current["file_name"])))]
        df_new_history = pd.concat([df_history, df_current], ignore_index=True)
    else:
        df_new_history = df_current

    df_new_history.to_csv(HISTORY_FILE, index=False)
    print(f"💾 Successfully saved and updated results in '{HISTORY_FILE}'.")
    
    # ==========================================
    # DISPLAY 1: Current Run Report (Formatted)
    # ==========================================
    print("\n📊 --- CURRENT RUN REPORT ---")
    display_df = df_current.copy()
    display_df["orig_size_bytes"] = display_df["orig_size_bytes"].map("{:,}".format)
    display_df["comp_size_bytes"] = display_df["comp_size_bytes"].map("{:,}".format)
    display_df["comp_ratio"] = display_df["comp_ratio"].map("{:.2f}x".format)
    display_df["space_saving_pct"] = display_df["space_saving_pct"].map("{:.2f}%".format)
    display(display_df[[
        "file_name", "orig_size_bytes", "comp_size_bytes", 
        "comp_time_ms", "decomp_time_ms", "comp_ratio", 
        "space_saving_pct", "comp_throughput_mbs", "decomp_throughput_mbs", "verification"
    ]])
    
    # ==========================================
    # DISPLAY 2: Cross-Version Historical Comparison
    # ==========================================
    print("\n📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---")
    # Reload full history to ensure data integrity
    df_full_history = pd.read_csv(HISTORY_FILE)
    
    # Pivot table to compare Space Saving (%) across versions for each file
    pivot_space = df_full_history.pivot_table(
        index="file_name", 
        columns="version", 
        values="space_saving_pct"
    )
    
    # Sort index to match target files order for consistency
    existing_targets = [f for f in TARGET_FILES if f in pivot_space.index]
    pivot_space = pivot_space.reindex(existing_targets)
    
    print("\n[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.")
    display(pivot_space.style.format("{:.2f}%").highlight_max(axis=1, color="lightgreen"))

# Execute the benchmark
run_benchmark_with_history()

🚀 Starting Benchmark for Version: v3_two_tree
📝 Notes: Version 3: Multi-stage pipeline upgraded to dual discrete Canonical Huffman Trees (Literal/Length and Distance) featuring tail-truncated headers.
🔄 Repeating each test 5 times to eliminate timing noise...

💾 Successfully saved and updated results in 'benchmark_history.csv'.

📊 --- CURRENT RUN REPORT ---


,file_name,orig_size_bytes,comp_size_bytes,comp_time_ms,decomp_time_ms,comp_ratio,space_saving_pct,comp_throughput_mbs,decomp_throughput_mbs,verification
0,test1.txt,35,290,0.41,0.21,0.12x,-728.57%,0.08,0.16,PASS
1,test2.txt,"2,638","3,954",2.91,3.02,0.67x,-49.89%,0.86,0.83,PASS
2,test3.txt,"5,349","6,252",5.60,5.69,0.86x,-16.88%,0.91,0.90,PASS
3,Lenna.bmp,"263,224","182,922",281.50,322.21,1.44x,30.51%,0.89,0.78,PASS
4,Cameraman.bmp,"66,616","48,647",74.31,82.73,1.37x,26.97%,0.85,0.77,PASS



📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---

[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.


version,v1_base,v2_base,v3_two_tree
file_name,,,
test1.txt,-3962.86%,-717.14%,-728.57%
test2.txt,-201.40%,24.49%,-49.89%
test3.txt,-103.12%,27.74%,-16.88%
Lenna.bmp,14.40%,18.80%,30.51%
Cameraman.bmp,-2.51%,14.77%,26.97%
